# Table creation


In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from utils import load_data, calculate_stability_metrics, bootstrap_confidence_intervals, stratified_bootstrap_ci, bootstrap_relative_instability

## 0. Setup


In [2]:
data = load_data("../data.parquet")

In [3]:
# Maps with correct display of names and ordering
STRATEGY_MAP = {
    "io": "IO",
    "cot": "CoT",
    "cot_sc": "CoT-SC",
    "react": "ReAct",
    "reflexion": "Reflexion",
    "tot_dfs": "ToT-DFS",
    "tot_bfs": "ToT-BFS",
    "got": "GoT",
    "rap": "RAP",
    "foa": "FoA"
}

MODEL_MAP ={
    'deepseek-ai/DeepSeek-R1': "DeepSeek-R1",
    'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': "Llama 4 Maverick",
    'gpt-4.1-mini': "GPT-4.1 Mini", 
    'gpt-4.1-nano': "GPT-4.1 Nano",
    'Qwen/Qwen3-235B-A22B-Thinking-2507': "Qwen3 235B Thinking", 
    'openai/gpt-oss-120b': "GPT-OSS 120B", 
    'gpt-5-mini': "GPT-5 Mini",
    'gpt-5-nano': "GPT-5 Nano",
    'claude-haiku-4-5-20251001': "Claude Haiku 4.5", 
    'gemini-3-flash-preview': "Gemini 3 Flash Preview",
}

In [4]:
def get_final_table(df: pd.DataFrame, mode: str, col: str):
    """
    Computes the final table with performance metrics, relative instability, and noise metrics for a given DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame containing the data.
        mode (str): The mode of analysis, either "Model" or "Strategy".
        col (str): The column name for which the metrics are computed.
    """

    # Computing the metrics
    # 1. Stability Analysis (Variance of Z-Scores)
    stability_df = calculate_stability_metrics(df, mode=mode, col=col)

    # 2. Confidence Intervals (Bootstrap)
    ci_df = bootstrap_confidence_intervals(df, mode=mode, col=col)

    # 3. Stratified Confidence Intervals (Bootstrap)
    strat_ci_df = stratified_bootstrap_ci(df, mode=mode ,col=col)

    # 4. Noise Evaluation
    rel_instability_df = bootstrap_relative_instability(df, mode=mode, col=col)

    # 5. Mean values
    raw_means = df.groupby(mode)[col].mean()

    # Creating tables for each group of metrics
    # Table 1: Performance
    table1 = pd.DataFrame({
        f"{col} (Avg)": raw_means,
        "95% CI (Stratified)": strat_ci_df["Strat_CI_Formatted"]
    })

    # Table 2: Relative Instability
    instability_pct = (rel_instability_df["Rel_Instability_Mean"] * 100).round(2).astype(str) + "%"
    table2 = pd.DataFrame({
        "Rel. Run Instability": instability_pct,
        "95% CI (Instability)": rel_instability_df["Rel_Instability_CI"]
    })

    # Table 3: Noise Metrics (Z-based)
    table3 = pd.DataFrame({
        "Global Noise (Z-Var)": stability_df["Global_Noise"],
        "Run Noise (Z-Var)": stability_df["Run_Noise"]
    })

    # Merging the tables into a final table
    table = pd.concat([table1, table2, table3], axis=1)
    table = table.reset_index()

    # Formatting the final table
    map = MODEL_MAP if mode == "Model" else STRATEGY_MAP
    table[mode] = table[mode].map(map)
    table[mode] = pd.Categorical(table[mode], categories=map.values(), ordered=True)
    table = table.sort_values(mode).reset_index(drop=True)


    return table

## 1. GPT-4.1 Nano (Strategies)


In [5]:
nano = data[data.Model=="gpt-4.1-nano"].copy()
score_table = get_final_table(nano, mode="Strategy", col="Score")
cost_table = get_final_table(nano, mode="Strategy", col="Cost")

Calculating Z-Score Variance (Stability) for 'Score'...
Bootstrapping Confidence Intervals for 'Score' (n=1000)...
Bootstrapping Stratified CIs for 'Score' (n=1000)...
Bootstrapping Relative Instability for 'Score' (n=1000)...
Strats: ['io' 'cot_sc' 'foa' 'cot' 'react' 'rap' 'got' 'tot_dfs' 'tot_bfs'
 'reflexion']
Calculating Z-Score Variance (Stability) for 'Cost'...
Bootstrapping Confidence Intervals for 'Cost' (n=1000)...
Bootstrapping Stratified CIs for 'Cost' (n=1000)...
Bootstrapping Relative Instability for 'Cost' (n=1000)...
Strats: ['io' 'cot_sc' 'foa' 'cot' 'react' 'rap' 'got' 'tot_dfs' 'tot_bfs'
 'reflexion']


### 1.1 Score Results


In [6]:
display(score_table)

,Strategy,Score (Avg),95% CI (Stratified),Rel. Run Instability,95% CI (Instability),Global Noise (Z-Var),Run Noise (Z-Var)
0,IO,0.134132,"[12.49, 14.43]",974.0%,"[5.05, 21.88]",0.140005,0.007140
1,CoT,0.279155,"[25.50, 30.31]",2642.61%,"[9.15, 55.86]",0.650253,0.098302
2,CoT-SC,0.226481,"[21.11, 24.32]",6580.7%,"[37.17, 263.14]",0.316806,0.032368
3,ReAct,0.303968,"[24.33, 34.33]",4036.25%,"[23.08, 86.28]",0.980934,0.314832
4,Reflexion,0.312413,"[25.33, 35.33]",4860.19%,"[24.42, 134.57]",0.981016,0.304610
5,ToT-DFS,0.094941,"[7.38, 11.72]",565.99%,"[1.07, 11.94]",1.450071,0.051675
6,ToT-BFS,0.414454,"[38.68, 44.23]",1149.26%,"[4.27, 21.86]",0.411390,0.109209
7,GoT,0.336361,"[30.91, 36.51]",1533.13%,"[6.01, 29.87]",0.479907,0.099805
8,RAP,0.371333,"[34.00, 40.33]",2127.78%,"[13.18, 44.16]",1.236621,0.062354
9,FoA,0.454870,"[43.05, 47.82]",757.05%,"[2.69, 15.50]",0.569975,0.096366


### 1.2 Cost Results


In [7]:
display(cost_table)

,Strategy,Cost (Avg),95% CI (Stratified),Rel. Run Instability,95% CI (Instability),Global Noise (Z-Var),Run Noise (Z-Var)
0,IO,0.005396,"[0.47, 0.60]",331.35%,"[1.26, 6.32]",0.008952,0.000005
1,CoT,0.013110,"[1.22, 1.41]",457.18%,"[2.19, 7.31]",0.009480,0.000015
2,CoT-SC,0.068256,"[6.75, 6.90]",80.75%,"[0.36, 1.28]",0.072363,0.000043
3,ReAct,0.062411,"[5.62, 7.06]",1005.21%,"[4.58, 18.47]",0.020516,0.001069
4,Reflexion,0.074986,"[6.68, 8.52]",1146.63%,"[5.02, 20.17]",0.030585,0.001856
5,ToT-DFS,0.103247,"[9.50, 11.11]",399.34%,"[1.76, 6.83]",0.387341,0.005656
6,ToT-BFS,0.436844,"[41.90, 45.23]",451.32%,"[2.06, 8.53]",1.123276,0.006055
7,GoT,0.494569,"[48.14, 50.83]",191.34%,"[0.85, 3.24]",1.390058,0.003453
8,RAP,0.534386,"[44.45, 69.27]",1629.84%,"[3.73, 36.95]",1.956214,0.358441
9,FoA,0.322831,"[31.53, 33.09]",344.91%,"[1.19, 6.86]",0.384149,0.001233


## 2. Across Models (Models)


In [8]:
models = data[data.Strategy=="io"].copy()
score_table = get_final_table(models, mode="Model", col="Score")
cost_table = get_final_table(models, mode="Model", col="Cost")

Calculating Z-Score Variance (Stability) for 'Score'...
Bootstrapping Confidence Intervals for 'Score' (n=1000)...
Bootstrapping Stratified CIs for 'Score' (n=1000)...
Bootstrapping Relative Instability for 'Score' (n=1000)...
Strats: ['Qwen/Qwen3-235B-A22B-Thinking-2507' 'openai/gpt-oss-120b'
 'deepseek-ai/DeepSeek-R1'
 'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8'
 'claude-haiku-4-5-20251001' 'gemini-3-flash-preview' 'gpt-4.1-mini'
 'gpt-5-nano' 'gpt-5-mini' 'gpt-4.1-nano']
Calculating Z-Score Variance (Stability) for 'Cost'...
Bootstrapping Confidence Intervals for 'Cost' (n=1000)...
Bootstrapping Stratified CIs for 'Cost' (n=1000)...
Bootstrapping Relative Instability for 'Cost' (n=1000)...
Strats: ['Qwen/Qwen3-235B-A22B-Thinking-2507' 'openai/gpt-oss-120b'
 'deepseek-ai/DeepSeek-R1'
 'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8'
 'claude-haiku-4-5-20251001' 'gemini-3-flash-preview' 'gpt-4.1-mini'
 'gpt-5-nano' 'gpt-5-mini' 'gpt-4.1-nano']


### 2.1 Score table


In [9]:
score_table

,Model,Score (Avg),95% CI (Stratified),Rel. Run Instability,95% CI (Instability),Global Noise (Z-Var),Run Noise (Z-Var)
0,DeepSeek-R1,0.232575,"[20.00, 26.42]",4350.21%,"[17.10, 83.67]",1.321076,0.064981
1,Llama 4 Maverick,0.409513,"[38.02, 43.90]",698.38%,"[1.45, 16.00]",0.325211,0.032434
2,GPT-4.1 Mini,0.456704,"[42.83, 48.56]",1121.97%,"[6.54, 17.13]",0.610863,0.018280
3,GPT-4.1 Nano,0.134132,"[12.50, 14.43]",983.56%,"[5.07, 21.89]",0.351011,0.002739
4,Qwen3 235B Thinking,0.352542,"[39.40, 43.19]",3583.62%,"[19.04, 88.48]",0.601279,0.031391
5,GPT-OSS 120B,0.501181,"[46.84, 53.74]",1129.6%,"[3.59, 21.43]",0.159857,0.062960
6,GPT-5 Mini,0.566020,"[53.28, 59.86]",968.41%,"[3.53, 17.84]",0.342221,0.054401
7,GPT-5 Nano,0.508460,"[48.66, 53.17]",1247.71%,"[4.24, 26.04]",0.688493,0.028635
8,Claude Haiku 4.5,0.377669,"[35.29, 40.50]",1078.2%,"[3.57, 19.52]",0.308582,0.042826
9,Gemini 3 Flash Preview,0.761522,"[74.00, 78.33]",377.34%,"[1.32, 6.85]",0.392869,0.049511


### 2.2 Cost table


In [10]:
cost_table

,Model,Cost (Avg),95% CI (Stratified),Rel. Run Instability,95% CI (Instability),Global Noise (Z-Var),Run Noise (Z-Var)
0,DeepSeek-R1,1.403321,"[119.46, 157.63]",1013.9%,"[5.78, 15.34]",1.072761,0.149130
1,Llama 4 Maverick,0.018653,"[1.79, 1.94]",292.27%,"[1.21, 5.11]",0.000628,0.000004
2,GPT-4.1 Mini,0.015175,"[1.18, 1.92]",944.19%,"[3.35, 29.07]",0.000513,0.000134
3,GPT-4.1 Nano,0.005396,"[0.47, 0.61]",335.97%,"[1.13, 6.23]",0.000595,0.000003
4,Qwen3 235B Thinking,0.651208,"[46.44, 67.79]",1233.35%,"[7.06, 17.45]",0.150962,0.055352
5,GPT-OSS 120B,0.028992,"[2.66, 3.17]",638.24%,"[3.10, 10.61]",0.001566,0.000067
6,GPT-5 Mini,0.167707,"[15.92, 17.89]",534.39%,"[2.10, 9.72]",0.102578,0.000666
7,GPT-5 Nano,0.058880,"[5.62, 6.13]",437.78%,"[2.00, 7.62]",0.009020,0.000053
8,Claude Haiku 4.5,0.111605,"[9.73, 12.88]",478.63%,"[1.05, 10.40]",0.006625,0.001866
9,Gemini 3 Flash Preview,0.977012,"[86.58, 103.40]",950.12%,"[4.43, 20.36]",1.025721,0.392051


## 3. GPT-4.1 Mini (Strategies)


In [11]:
mini = data[data.Model=="gpt-4.1-mini"].copy()
score_table = get_final_table(mini, mode="Strategy", col="Score")
cost_table = get_final_table(mini, mode="Strategy", col="Cost")

Calculating Z-Score Variance (Stability) for 'Score'...
Bootstrapping Confidence Intervals for 'Score' (n=1000)...
Bootstrapping Stratified CIs for 'Score' (n=1000)...
Bootstrapping Relative Instability for 'Score' (n=1000)...
Strats: ['io' 'cot_sc' 'foa' 'cot' 'react' 'rap' 'got' 'tot_dfs' 'tot_bfs'
 'reflexion']
Calculating Z-Score Variance (Stability) for 'Cost'...
Bootstrapping Confidence Intervals for 'Cost' (n=1000)...
Bootstrapping Stratified CIs for 'Cost' (n=1000)...
Bootstrapping Relative Instability for 'Cost' (n=1000)...
Strats: ['io' 'cot_sc' 'foa' 'cot' 'react' 'rap' 'got' 'tot_dfs' 'tot_bfs'
 'reflexion']


### 3.1 Score table


In [12]:
score_table

,Strategy,Score (Avg),95% CI (Stratified),Rel. Run Instability,95% CI (Instability),Global Noise (Z-Var),Run Noise (Z-Var)
0,IO,0.456704,"[42.89, 48.81]",1111.4%,"[6.69, 16.73]",0.376652,0.020728
1,CoT,0.436301,"[41.36, 45.91]",870.25%,"[3.07, 16.75]",0.285346,0.014804
2,CoT-SC,0.384761,"[35.80, 41.47]",1898.46%,"[11.52, 38.82]",0.418543,0.030377
3,ReAct,0.420079,"[37.67, 47.67]",2484.83%,"[8.47, 42.97]",1.858736,0.230177
4,Reflexion,0.426413,"[38.00, 48.67]",3721.65%,"[14.19, 76.59]",1.845718,0.225561
5,ToT-DFS,0.209719,"[18.38, 23.72]",271.77%,"[0.63, 5.25]",2.219202,0.012567
6,ToT-BFS,0.520375,"[48.55, 58.04]",1937.2%,"[4.90, 59.76]",0.636212,0.101371
7,GoT,0.446754,"[41.41, 49.08]",2572.49%,"[9.41, 63.24]",0.398415,0.067050
8,RAP,0.324698,"[28.93, 37.23]",2714.6%,"[14.80, 44.65]",0.573839,0.125143
9,FoA,0.520367,"[47.49, 57.41]",2169.02%,"[7.37, 36.32]",0.505578,0.147924


### 3.2 Cost table


In [13]:
cost_table

,Strategy,Cost (Avg),95% CI (Stratified),Rel. Run Instability,95% CI (Instability),Global Noise (Z-Var),Run Noise (Z-Var)
0,IO,0.015175,"[1.16, 1.92]",923.31%,"[3.28, 27.49]",0.005959,0.000016
1,CoT,0.062690,"[5.55, 6.99]",548.54%,"[2.18, 10.12]",0.008685,0.000048
2,CoT-SC,0.273652,"[27.12, 27.66]",62.94%,"[0.26, 1.13]",0.105751,0.000060
3,ReAct,0.234146,"[22.45, 24.81]",548.06%,"[2.61, 10.17]",0.030648,0.000170
4,Reflexion,0.293067,"[27.43, 31.51]",762.09%,"[3.70, 13.25]",0.046623,0.000625
5,ToT-DFS,0.380443,"[35.89, 40.69]",251.04%,"[0.71, 10.41]",0.359350,0.002375
6,ToT-BFS,1.704855,"[164.25, 176.65]",369.84%,"[1.78, 6.41]",1.088346,0.006186
7,GoT,1.500060,"[137.16, 191.81]",624.72%,"[1.53, 24.07]",0.914882,0.082350
8,RAP,2.423972,"[228.28, 260.36]",599.81%,"[3.23, 9.17]",1.056948,0.042802
9,FoA,1.249754,"[121.18, 134.01]",335.01%,"[1.36, 5.72]",0.627955,0.004214


## 4. Llama 4 Maverick (Strategies)


In [14]:
llama = data[data.Model=="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"].copy()
score_table = get_final_table(llama, mode="Strategy", col="Score")
cost_table = get_final_table(llama, mode="Strategy", col="Cost")

Calculating Z-Score Variance (Stability) for 'Score'...
Bootstrapping Confidence Intervals for 'Score' (n=1000)...
Bootstrapping Stratified CIs for 'Score' (n=1000)...
Bootstrapping Relative Instability for 'Score' (n=1000)...
Strats: ['io' 'react' 'cot' 'rap' 'got' 'foa' 'cot_sc' 'tot_dfs' 'tot_bfs'
 'reflexion']
Calculating Z-Score Variance (Stability) for 'Cost'...
Bootstrapping Confidence Intervals for 'Cost' (n=1000)...
Bootstrapping Stratified CIs for 'Cost' (n=1000)...
Bootstrapping Relative Instability for 'Cost' (n=1000)...
Strats: ['io' 'react' 'cot' 'rap' 'got' 'foa' 'cot_sc' 'tot_dfs' 'tot_bfs'
 'reflexion']


### 4.1 Score table


In [15]:
score_table

,Strategy,Score (Avg),95% CI (Stratified),Rel. Run Instability,95% CI (Instability),Global Noise (Z-Var),Run Noise (Z-Var)
0,IO,0.409513,"[38.13, 43.98]",690.34%,"[1.62, 15.34]",0.412371,0.185553
1,CoT,0.410500,"[36.50, 45.50]",1278.31%,"[4.56, 25.15]",0.173164,0.065162
2,CoT-SC,0.350288,"[32.09, 37.63]",2187.86%,"[10.88, 44.03]",0.200181,0.025354
3,ReAct,0.307849,"[27.98, 34.71]",1033.23%,"[4.30, 24.03]",0.890581,0.095523
4,Reflexion,0.332421,"[30.38, 38.25]",1923.92%,"[10.27, 32.35]",1.079482,0.113785
5,ToT-DFS,0.095000,"[6.50, 14.00]",2210.06%,"[1.02, 66.02]",1.590458,0.133655
6,ToT-BFS,0.617017,"[48.54, 56.63]",649.23%,"[2.36, 10.22]",0.531138,0.121100
7,GoT,0.421573,"[38.70, 47.38]",2647.77%,"[6.65, 55.49]",0.684502,0.142023
8,RAP,0.183600,"[11.60, 21.60]",1286.15%,"[4.60, 27.60]",3.621772,0.295240
9,FoA,0.597041,"[52.79, 58.87]",601.53%,"[3.64, 10.45]",1.375127,0.119180


### 4.2 Cost table


In [16]:
cost_table

,Strategy,Cost (Avg),95% CI (Stratified),Rel. Run Instability,95% CI (Instability),Global Noise (Z-Var),Run Noise (Z-Var)
0,IO,0.018653,"[1.79, 1.94]",295.3%,"[1.15, 5.16]",0.211983,0.166667
1,CoT,0.029291,"[2.79, 3.14]",480.42%,"[1.62, 8.34]",0.009819,0.000008
2,CoT-SC,0.205323,"[20.28, 20.82]",88.56%,"[0.32, 1.59]",0.031579,0.000044
3,ReAct,0.185480,"[17.83, 19.49]",287.09%,"[1.00, 5.03]",0.013645,0.000201
4,Reflexion,0.232452,"[21.94, 24.80]",517.54%,"[2.32, 8.66]",0.021986,0.000833
5,ToT-DFS,0.389778,"[34.70, 43.32]",374.67%,"[1.33, 7.33]",1.100351,0.012391
6,ToT-BFS,0.930954,"[100.08, 108.00]",593.6%,"[2.27, 12.80]",0.221543,0.007374
7,GoT,1.388569,"[131.95, 140.87]",181.49%,"[0.64, 3.37]",1.853360,0.003011
8,RAP,2.569056,"[243.61, 267.03]",292.66%,"[1.22, 6.86]",0.557644,0.019025
9,FoA,0.886235,"[90.60, 94.97]",598.6%,"[1.87, 12.05]",0.454482,0.001361
